<h2 style="color:#2E86C1; text-align:center;">
Analysis of Security Inspection Performance and Risk Trends Across Healthcare Facilities - Data Preprocessing
</h2>

### Objective
Prepare a refined dataset optimized for analysis and modeling by cleaning, transforming, and enhancing the raw inspection dataset.

### Import Required Libraries
Load essential libraries for data manipulation and numerical operations.
- `pandas` → Data manipulation
- `numpy` → Numerical operations

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported successfully!')

Libraries imported successfully!


### Load Dataset
Load the raw Security Inspection dataset for preprocessing.

In [14]:
df = pd.read_excel('Security_Inspection Data_16400x18.xlsx')
print('Dataset Shape:', df.shape)
df.head()

Dataset Shape: (16400, 18)


,Facility_ID,Facility_Name,Region,Department,Inspection_Type,Risk_Category,Inspection_Date,Inspector_ID,Guard_ID,Non_Compliance_Count,Staff_On_Duty,Incident_Reported,Inspection_Score,Uniform_Grooming_Score,Knowledge_Check_Score,Corrective_Action_Required,Follow_Up_Status,Remarks
0,1025,Medical Center,South,Admin,Spot Inspection,Medium,23/08/2025,301,5380,3,9,Yes,75,79,68.00,Yes,Closed,Guard briefed
1,1067,Medical Center,East,Radiology,Scheduled Audit,High,24/07/2025,275,7818,1,44,No,79,85,74.00,No,Closed,Needs refresher training
2,1103,General Hospital,West,ICU,Scheduled Audit,Low,19/04/2024,273,5126,2,11,No,85,77,77.00,No,Open,All OK
3,1092,General Hospital,South,OPD,Re-Audit,Low,2024-06-17,289,8892,5,20,No,93,58,84.00,No,In Progress,Excellent compliance
4,1142,Community Hospital,Central,ICU,Re-Audit,Low,15/08/2025,283,8410,0,49,No,83,100,70.00,No,Closed,Follow-up required


### Check Missing Values
 Identify columns with missing data that may affect analysis or modeling.

In [3]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

Department                304
Inspection_Date           146
Incident_Reported         508
Uniform_Grooming_Score    644
Knowledge_Check_Score     819
Remarks                   971
dtype: int64

### Handle Missing Values
- Fill numerical columns with median values
- Fill categorical columns with mode values
This ensures no null values remain in the dataset.

In [5]:
# Fill numerical columns with median
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical columns with mode
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum().sum()

np.int64(1545)

### Create Performance Category
 Classify inspection scores into performance levels:
- Excellent (≥85)
- Good (70–84)
- Average (50–69)
- Poor (<50)

This helps in categorical analysis and predictive modeling.

In [12]:
df['Inspection_Score'] = pd.to_numeric(df['Inspection_Score'], errors='coerce')

conditions = [
    df['Inspection_Score'] >= 85,
    df['Inspection_Score'] >= 70,
    df['Inspection_Score'] >= 50
]

choices = ['Excellent', 'Good', 'Average']

df['Performance_Category'] = np.select(conditions, choices, default='Poor')

df[['Inspection_Score', 'Performance_Category']].head()

,Inspection_Score,Performance_Category
0,75.00,Good
1,79.00,Good
2,85.00,Excellent
3,93.00,Excellent
4,83.00,Good


### Create Compliance Rate

Create a new feature representing compliance level based on non-compliance count.

Compliance Rate = 100 − (Non_Compliance_Count × 10)

Values are clipped at 0 to avoid negative percentages.

In [7]:
df['Compliance_Rate'] = 100 - (df['Non_Compliance_Count'] * 10)
df['Compliance_Rate'] = df['Compliance_Rate'].clip(lower=0)

df[['Non_Compliance_Count', 'Compliance_Rate']].head()

,Non_Compliance_Count,Compliance_Rate
0,3,70
1,1,90
2,2,80
3,5,50
4,0,100


### Export Cleaned Dataset

In [11]:
df.to_excel('Cleaned_Security_Inspection_Data.xlsx', index=False)
print('Cleaned dataset exported successfully!')

Cleaned dataset exported successfully!
